In [ ]:
import subprocess
subprocess.run(["pip", "install", "numpy==1.26.4", "-q"])
import os
os.kill(os.getpid(), 9)

In [ ]:
# ── Cell 1 (after restart) ─────────────────────────────────────────────
!pip install numpy==1.26.4 -q
!pip install transformers==4.40.0 datasets peft accelerate \
             torch torchvision tensorflow==2.15.0 keras \
             scikit-learn seaborn matplotlib gradio \
             arabert --quiet

import numpy as np
print("numpy:", np.__version__)          # should be 1.26.4

import torch
print("PyTorch:", torch.__version__)

import tensorflow as tf
print("TensorFlow:", tf.__version__)

ERROR: Could not find a version that satisfies the requirement tensorflow==2.15.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow==2.15.0
numpy: 1.26.4
PyTorch: 2.10.0+cu128
TensorFlow: 2.19.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import os


# SOURCE DOMAIN: mksaad MSA Arabic tweets
src_train = pd.read_csv('/content/drive/MyDrive/NLP/Arabic Tweets - sentiment/train_Arabic_tweets_positive_negative.tsv',
                         sep='\t', header=None, names=['label', 'text'])
src_test  = pd.read_csv('/content/drive/MyDrive/NLP/Arabic Tweets - sentiment/test_Arabic_tweets_positive_negative.tsv',
                         sep='\t', header=None, names=['label', 'text'])
src_df = pd.concat([src_train, src_test], ignore_index=True)

print("=== SOURCE: mksaad ===")
print("Shape:", src_df.shape)
print("Labels:", src_df['label'].value_counts().to_dict())
print("Sample:", src_df['text'].iloc[0])
print()

# TARGET DOMAIN: Raïdy Arabizi 3-class
tgt_df = pd.read_csv('/content/drive/MyDrive/NLP/Arabizi Tweets/3-class-sentiment-arabizi-ds.csv')

print("=== TARGET: Raïdy Arabizi ===")
print("Shape:", tgt_df.shape)
print("Columns:", tgt_df.columns.tolist())
print("Labels:", tgt_df.iloc[:, -1].value_counts().to_dict())
print("Sample:", tgt_df.iloc[0, 0])
print()

# Check null values in both
print("Source nulls:", src_df.isnull().sum().to_dict())
print("Target nulls:", tgt_df.isnull().sum().to_dict())

=== SOURCE: mksaad ===
Shape: (56795, 2)
Labels: {'pos': 28513, 'neg': 28282}
Sample: نحن الذين يتحول كل ما نود أن نقوله إلى دعاء لله، لا تبحثوا فينا عن قوة، إننا مكسورون، القوة التي…

=== TARGET: Raïdy Arabizi ===
Shape: (1799, 3)
Columns: ['tweet', 'sentiment', 'highlight']
Labels: {'Sarcasm': 101, 'Joke': 83, 'Bullying': 44, 'Foul language': 21, 'Saying': 18, 'Known fact': 17, 'Courtesy words': 16, 'Sectarianism': 13, 'Sexism': 2, 'Racism': 1}
Sample: Aw enn l ahla men hek hay li btelbesle crop top b noss din l sa23a w l talej w bte23ad tne2 "Msa23aaa" "Ya alla ktir sa23aa" N2ebre lbese. 

Source nulls: {'label': 0, 'text': 0}
Target nulls: {'tweet': 0, 'sentiment': 0, 'highlight': 1483}


In [ ]:
tgt_df = tgt_df.drop(columns=['highlight'])

In [ ]:
SRC_LABEL_MAP = {'positive': 1, 'negative': 0}

print("Unique target labels:", tgt_df.iloc[:, -1].unique())
TGT_LABEL_MAP = {'Positive': 2, 'Neutral': 1, 'Negative': 0}
print("✓ Setup complete")

Unique target labels: ['Negative' 'Neutral' 'Positive']
✓ Setup complete


In [ ]:
tgt_df.head(10)

,tweet,sentiment
0,Aw enn l ahla men hek hay li btelbesle crop to...,Negative
1,yu2brnee jamelo pepe :p tfeh shu beche3 bas li...,Negative
2,Lea ktir pedophile 😂,Negative
3,Shu hal hmar hayda,Negative
4,Fasharet 3a ra2btak w ra2bit m3almak w ra2bit ...,Negative
5,Hayda ensen marid,Negative
6,my uncle is lowkey aawne w ktir aabele natfo,Negative
7,Bassam bte7lam feyon lal syesye bel leil as a ...,Negative
8,@Ritarouhana helo ktir. Eza badkoun tejo kelko...,Negative
9,Hayda ma biya3ref ye2ra w yektob w bedo wazire...,Negative


In [ ]:
import re

def clean_msa_arabic(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|@\w+|#\w+', '', text)
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F]', '', text)
    text = re.sub(r'[أإآٱ]', 'ا', text)
    text = re.sub(r'[ىئ]', 'ي', text)
    text = re.sub(r'\u0640', '', text)
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

SRC_LABEL_MAP = {'pos': 1, 'neg': 0}

src_df['label'] = src_df['label'].astype(str).str.strip().str.lower()
src_df['clean_text'] = src_df['text'].apply(clean_msa_arabic)
src_df['label_id'] = src_df['label'].map(SRC_LABEL_MAP)

src_df = src_df.dropna(subset=['clean_text', 'label_id'])
src_df = src_df[src_df['clean_text'].str.len() > 5]

print(f"Source after cleaning: {len(src_df)} rows")
print(src_df['label_id'].value_counts())
print(src_df[['text', 'clean_text', 'label', 'label_id']].head())

Source after cleaning: 56497 rows
label_id
1    28381
0    28116
Name: count, dtype: int64
                                                text  \
0  نحن الذين يتحول كل ما نود أن نقوله إلى دعاء لل...   
1  وفي النهاية لن يبقىٰ معك آحدإلا من رأىٰ الجمال...   
2                                    من الخير نفسه 💛   
3  #زلزل_الملعب_نصرنا_بيلعب كن عالي الهمه ولا ترض...   
4  الشيء الوحيد الذي وصلوا فيه للعالمية هو : المس...   

                                          clean_text label  label_id  
0  نحن الذين يتحول كل ما نود ان نقوله الي دعاء لل...   pos         1  
1  وفي النهاية لن يبقيٰ معك احدالا من رايٰ الجمال...   pos         1  
2                                      من الخير نفسه   pos         1  
3  كن عالي الهمه ولا ترضي بغير القمه مجرد ساعات ل...   pos         1  
4  الشيء الوحيد الذي وصلوا فيه للعالمية هو المسيا...   pos         1  


In [ ]:
src_df.head(10)

,label,text,clean_text,label_id
0,pos,نحن الذين يتحول كل ما نود أن نقوله إلى دعاء لل...,نحن الذين يتحول كل ما نود ان نقوله الي دعاء لل...,1
1,pos,وفي النهاية لن يبقىٰ معك آحدإلا من رأىٰ الجمال...,وفي النهاية لن يبقيٰ معك احدالا من رايٰ الجمال...,1
2,pos,من الخير نفسه 💛,من الخير نفسه,1
3,pos,#زلزل_الملعب_نصرنا_بيلعب كن عالي الهمه ولا ترض...,كن عالي الهمه ولا ترضي بغير القمه مجرد ساعات ل...,1
4,pos,الشيء الوحيد الذي وصلوا فيه للعالمية هو : المس...,الشيء الوحيد الذي وصلوا فيه للعالمية هو المسيا...,1
5,pos,#الاتحاد_النصر لاتحسبونا نسينا يالطواقي ولانبي...,لاتحسبونا نسينا يالطواقي ولانبيكم توقفون معنا ...,1
6,pos,احبك انت وياه واموري من سعه 🎶,احبك انت وياه واموري من سعه,1
7,pos,#تأمل قال الله ﷻ :- _*​﴿بواد غير ذي زرع ﴾*_ 💫💫...,قال الله بواد غير ذي زرع ومع ذلك هتف بالدعاء و...,1
8,pos,وينهم الي يرقصوا مع زخات المطر 💃 خلونا نشوفكم ...,وينهم الي يرقصوا مع زخات المطر خلونا نشوفكم لا...,1
9,pos,اللهم آمين يارب العالمين انتي وانا وامة سيدنا ...,اللهم امين يارب العالمين انتي وانا وامة سيدنا ...,1


In [ ]:
def clean_arabizi(text):
    """
    Cleans Lebanese Arabizi Twitter text.
    Apply to Raïdy dataset (target domain) ONLY.
    Key challenge: Arabizi uses numbers as letters — must handle carefully.
    """
    if not isinstance(text, str):
        return ""
    # 1. Remove URLs, mentions, hashtags
    text = re.sub(r'http\S+|@\w+|#\w+', '', text)
    # 2. Lowercase (Arabizi is case-insensitive)
    text = text.lower()
    # 3. OOV Strategy: map Arabizi digit-letters to Arabic Unicode
    # This converts "3arabi" → "عarabi" so mBERT sees familiar chars
    arabizi_map = {
        '3': 'ع',   # ain
        '2': 'ء',   # hamza
        '7': 'ح',   # ha
        '5': 'خ',   # kha
        '6': 'ط',   # ta
        '8': 'غ',   # ghayn
        '9': 'ق',   # qaf
        '4': 'ذ',   # thal
    }
    # Only replace digit if surrounded by letters (not standalone numbers)
    for digit, letter in arabizi_map.items():
        text = re.sub(rf'(?<=[a-zA-Z\u0600-\u06FF]){digit}|{digit}(?=[a-zA-Z\u0600-\u06FF])',
                      letter, text)
    # 4. Remove remaining punctuation (keep letters, Arabic chars, spaces)
    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)
    # 5. Collapse whitespace
    return re.sub(r'\s+', ' ', text).strip()

# Identify the correct text and label column names after inspection
TEXT_COL = tgt_df.columns[0]   # adjust after inspecting
LABEL_COL = tgt_df.columns[-1]  # adjust after inspecting

tgt_df['clean_text'] = tgt_df[TEXT_COL].apply(clean_arabizi)
tgt_df['label_id'] = tgt_df[LABEL_COL].map(TGT_LABEL_MAP)
tgt_df = tgt_df.dropna(subset=['clean_text', 'label_id'])
tgt_df['label_id'] = tgt_df['label_id'].astype(int)

print(f"Target after cleaning: {len(tgt_df)} rows")
print(tgt_df['label_id'].value_counts())

Target after cleaning: 1799 rows
label_id
0    600
2    600
1    599
Name: count, dtype: int64


In [ ]:
print(tgt_df[['tweet', 'sentiment']].head(20).to_string())

                                                                                                                                           tweet sentiment
0    Aw enn l ahla men hek hay li btelbesle crop top b noss din l sa23a w l talej w bte23ad tne2 "Msa23aaa" "Ya alla ktir sa23aa" N2ebre lbese.   Negative
1                                                                    yu2brnee jamelo pepe :p tfeh shu beche3 bas li kanze por la decima ma7leto   Negative
2                                                                                                                          Lea ktir pedophile 😂   Negative
3                                                                                                                            Shu hal hmar hayda   Negative
4                                                                                    Fasharet 3a ra2btak w ra2bit m3almak w ra2bit li khalafouk   Negative
5                                                                     

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

def oov_rate(texts, label=""):
    total_tokens, unk_tokens = 0, 0
    for text in texts[:500]:  # sample 500
        ids = tokenizer.encode(text, add_special_tokens=False)
        decoded = tokenizer.convert_ids_to_tokens(ids)
        total_tokens += len(decoded)
        unk_tokens += decoded.count('[UNK]')
    rate = unk_tokens / max(total_tokens, 1) * 100
    print(f"{label}: {unk_tokens}/{total_tokens} OOV tokens ({rate:.2f}%)")
    return rate

oov_msa    = oov_rate(src_df['clean_text'].tolist(), "MSA (before clean)")
oov_arb    = oov_rate(tgt_df[TEXT_COL].tolist(),       "Arabizi (raw)")
oov_arb_cl = oov_rate(tgt_df['clean_text'].tolist(), "Arabizi (after clean)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

MSA (before clean): 25/12881 OOV tokens (0.19%)
Arabizi (raw): 124/13501 OOV tokens (0.92%)
Arabizi (after clean): 1/11265 OOV tokens (0.01%)


In [ ]:
from transformers import MarianMTModel, MarianTokenizer
import torch

# Use English as pivot: Arabizi → EN → paraphrased back
# (Arabizi → AR would need a dedicated transliteration model)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")  # should print 'cuda'

en_ar = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-ar")
ar_en = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ar-en")
en_ar_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-ar")
ar_en_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-ar-en")

def translate_batch(texts, tok, model, max_len=128):
    inputs = tok(texts, return_tensors="pt", padding=True,
                 truncation=True, max_length=max_len)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_len)
    return tok.batch_decode(out, skip_special_tokens=True)

def back_translate_en_ar_en(texts):
    # EN → AR → EN: creates paraphrases of English portions
    ar = translate_batch(texts, en_ar, en_ar_model)
    back = translate_batch(ar, ar_en, ar_en_model)
    return back

# Augment ONLY the neutral class (label_id == 1)
neutral_texts = tgt_df[tgt_df['label_id'] == 1]['clean_text'].tolist()
print(f"Neutral samples before augmentation: {len(neutral_texts)}")

augmented = []
for i in range(0, len(neutral_texts), 32):
    batch = neutral_texts[i:i+32]
    augmented.extend(back_translate_en_ar_en(batch))

aug_df = pd.DataFrame({'clean_text': augmented, 'label_id': 1})
tgt_augmented = pd.concat([tgt_df[['clean_text','label_id']], aug_df],
                            ignore_index=True)

print("Label distribution after augmentation:")
print(tgt_augmented['label_id'].value_counts())

Using: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Neutral samples before augmentation: 599
Label distribution after augmentation:
label_id
1    1198
0     600
2     600
Name: count, dtype: int64


In [ ]:
import os, pandas as pd

BASE = '/content/drive/MyDrive/NLP'
os.makedirs(f'{BASE}/data/source', exist_ok=True)
os.makedirs(f'{BASE}/data/target', exist_ok=True)

# The important one — augmented data took the longest to create
tgt_augmented.to_csv(f'{BASE}/data/target/tgt_augmented.csv', index=False)
src_df.to_csv(f'{BASE}/data/source/src_df.csv',               index=False)
tgt_df.to_csv(f'{BASE}/data/target/tgt_df.csv',               index=False)

print(f"✓ tgt_augmented: {len(tgt_augmented)} rows saved")
print(f"✓ src_df: {len(src_df)} rows saved")
print(f"✓ tgt_df: {len(tgt_df)} rows saved")

✓ tgt_augmented: 2398 rows saved
✓ src_df: 56497 rows saved
✓ tgt_df: 1799 rows saved


In [ ]:
try:
    src_train_df.to_csv(f'{BASE}/data/source/train.csv', index=False)
    tgt_train_df.to_csv(f'{BASE}/data/target/train.csv', index=False)
    tgt_val_df.to_csv(f'{BASE}/data/target/val.csv',     index=False)
    tgt_test_df.to_csv(f'{BASE}/data/target/test.csv',   index=False)
    print("✓ Splits saved")
except NameError:
    print("Splits not created yet — that's fine, tgt_augmented is enough")

✓ Splits saved


In [ ]:
from sklearn.model_selection import train_test_split

# ── Source domain split (80/20 — we only need train here) ───
src_train_df, src_val_df = train_test_split(
    src_df[['clean_text', 'label_id']],
    test_size=0.1, stratify=src_df['label_id'], random_state=42
)
print(f"Source → train:{len(src_train_df)} | val:{len(src_val_df)}")

# ── Target domain split (70/15/15) ──────────────────────────
tgt_train_temp, tgt_test_df = train_test_split(
    tgt_augmented, test_size=0.15,
    stratify=tgt_augmented['label_id'], random_state=42
)
tgt_train_df, tgt_val_df = train_test_split(
    tgt_train_temp, test_size=0.176,  # 0.176 of 85% ≈ 15% total
    stratify=tgt_train_temp['label_id'], random_state=42
)
print(f"Target → train:{len(tgt_train_df)} | val:{len(tgt_val_df)} | test:{len(tgt_test_df)}")

# Save all to Drive
base = '/content/drive/MyDrive/NLP/Arabizi Tweets'
os.makedirs(f'{base}/source', exist_ok=True)
os.makedirs(f'{base}/target', exist_ok=True)
src_train_df.to_csv(f'{base}/source/train.csv', index=False)
tgt_train_df.to_csv(f'{base}/target/train.csv', index=False)
tgt_val_df.to_csv(f'{base}/target/val.csv',   index=False)
tgt_test_df.to_csv(f'{base}/target/test.csv',  index=False)
print("✓ All splits saved")

Source → train:50847 | val:5650
Target → train:1679 | val:359 | test:360
✓ All splits saved


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_NAME = "bert-base-multilingual-cased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load with 2-class head for SOURCE domain training first
src_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2  # binary: pos/neg for mksaad
).to(device)

total_params = sum(p.numel() for p in src_model.parameters())
trainable_params = sum(p.numel() for p in src_model.parameters()
                       if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model on device:      {device}")
# ↑ Save this number — you'll compare it to LoRA's 0.33% later

# Quick tokenization test on both scripts
msa_ex  = "هذا المنتج رائع جداً وأنا سعيد"
arb_ex  = "hatha el mante3 ra2i3 ktir w ana mabsout"

for ex, label in [(msa_ex, "MSA"), (arb_ex, "Arabizi")]:
    tokens = tokenizer.tokenize(ex)
    unk = tokens.count('[UNK]')
    print(f"{label}: {len(tokens)} tokens, {unk} UNK → {tokens[:8]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters:     177,854,978
Trainable parameters: 177,854,978
Model on device:      cuda
MSA: 11 tokens, 0 UNK → ['هذا', 'ال', '##من', '##تج', 'را', '##ئع', 'جدا', '##ً']...
Arabizi: 17 tokens, 0 UNK → ['hat', '##ha', 'el', 'man', '##te', '##3', 'ra', '##2']...


In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

class ArabicTweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

# ── Source domain loaders ───────────────────────────────────
src_train_ds = ArabicTweetDataset(
    src_train_df['clean_text'], src_train_df['label_id'], tokenizer
)
src_val_ds = ArabicTweetDataset(
    src_val_df['clean_text'], src_val_df['label_id'], tokenizer
)
src_train_loader = DataLoader(src_train_ds, batch_size=32, shuffle=True)
src_val_loader   = DataLoader(src_val_ds,   batch_size=64, shuffle=False)
print(f"Source batches: {len(src_train_loader)} train, {len(src_val_loader)} val")

Source batches: 1589 train, 89 val


In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
import time

EPOCHS_SRC = 2  # 2 epochs is enough on 58K samples
LR = 2e-5

optimizer = AdamW(src_model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=len(src_train_loader),
    num_training_steps=len(src_train_loader) * EPOCHS_SRC
)

src_start = time.time()

for epoch in range(EPOCHS_SRC):
    src_model.train()
    epoch_loss = 0

    for batch in src_train_loader:
        optimizer.zero_grad()
        outputs = src_model(
            input_ids      = batch['input_ids'].to(device),
            attention_mask = batch['attention_mask'].to(device),
            labels         = batch['labels'].to(device)
        )
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(src_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        epoch_loss += outputs.loss.item()

    # Validation
    src_model.eval()
    preds, true = [], []
    with torch.no_grad():
        for b in src_val_loader:
            out = src_model(input_ids=b['input_ids'].to(device),
                            attention_mask=b['attention_mask'].to(device))
            preds.extend(out.logits.argmax(-1).cpu().tolist())
            true.extend(b['labels'].tolist())

    f1 = f1_score(true, preds, average='macro')
    avg_loss = epoch_loss / len(src_train_loader)
    print(f"[Source] Epoch {epoch+1}/{EPOCHS_SRC} | Loss: {avg_loss:.4f} | F1: {f1:.4f}")

src_time = time.time() - src_start
print(f"Source domain training time: {src_time:.0f}s")

# ── Save this checkpoint — it's the transfer starting point ─
src_model.save_pretrained('/content/drive/MyDrive/NLP/Arabizi Tweets/models')
tokenizer.save_pretrained('/content/drive/MyDrive/NLP/Arabizi Tweets/models')
print("✓ Source model saved — this is your transfer learning starting point")

[Source] Epoch 1/2 | Loss: 0.5960 | F1: 0.7431
[Source] Epoch 2/2 | Loss: 0.4564 | F1: 0.7759
Source domain training time: 2414s
✓ Source model saved — this is your transfer learning starting point


In [ ]:
from transformers import AutoModelForSequenceClassification

# ── Load the source-domain-trained checkpoint ───────────────
# ignore_mismatched_sizes=True allows head dimension change 2→3
tgt_model = AutoModelForSequenceClassification.from_pretrained(
    '/content/drive/MyDrive/NLP/Arabizi Tweets/models',
    num_labels=3,                 # 3-class: neg/neutral/pos
    ignore_mismatched_sizes=True  # critical — replaces 2-class head
).to(device)

# Verify: count frozen vs trainable layers (all trainable = full FT)
frozen = sum(1 for p in tgt_model.parameters() if not p.requires_grad)
trainable = sum(1 for p in tgt_model.parameters() if p.requires_grad)
print(f"Full FT — Frozen layers: {frozen} | Trainable layers: {trainable}")
# → In full FT, frozen should be 0

# ── Also create a ZERO-TRANSFER baseline for comparison ─────
# This is mBERT with no source training — your "without transfer" number
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased", num_labels=3
).to(device)
print("✓ Zero-transfer baseline also loaded")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /content/drive/MyDrive/NLP/Arabizi Tweets/models and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([3]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([3, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Full FT — Frozen layers: 0 | Trainable layers: 201


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Zero-transfer baseline also loaded


In [ ]:
# ── Sanity check before training — confirm GPU memory is allocated ──────
import torch
print(f"tgt_model on:      {next(tgt_model.parameters()).device}")
print(f"baseline on:       {next(baseline_model.parameters()).device}")
allocated = torch.cuda.memory_allocated() / 1024**3
print(f"GPU RAM used:      {allocated:.2f} GB")
# Should show ~2.6GB (two mBERT models loaded)
# If it shows cuda:0 for both — proceed to training

tgt_model on:      cuda:0
baseline on:       cuda:0
GPU RAM used:      4.07 GB


In [ ]:
# ── Target domain DataLoaders ────────────────────────────────
tgt_train_ds = ArabicTweetDataset(
    tgt_train_df['clean_text'], tgt_train_df['label_id'], tokenizer
)
tgt_val_ds = ArabicTweetDataset(
    tgt_val_df['clean_text'], tgt_val_df['label_id'], tokenizer
)
tgt_train_loader = DataLoader(tgt_train_ds, batch_size=16, shuffle=True)
tgt_val_loader   = DataLoader(tgt_val_ds,   batch_size=32, shuffle=False)

# ── Training function (reuse for baseline too) ───────────────
def train_target(model, train_loader, val_loader, epochs=3, lr=2e-5, tag=""):
    opt = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(
        opt, num_warmup_steps=50,
        num_training_steps=len(train_loader)*epochs
    )
    start = time.time()
    history = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch in train_loader:
            opt.zero_grad()
            out = model(input_ids=batch['input_ids'].to(device),
                        attention_mask=batch['attention_mask'].to(device),
                        labels=batch['labels'].to(device))
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            epoch_loss += out.loss.item()

        model.eval()
        preds, true = [], []
        with torch.no_grad():
            for b in val_loader:
                o = model(input_ids=b['input_ids'].to(device),
                          attention_mask=b['attention_mask'].to(device))
                preds.extend(o.logits.argmax(-1).cpu().tolist())
                true.extend(b['labels'].tolist())
        f1 = f1_score(true, preds, average='macro')
        history.append({'epoch': epoch+1, 'f1': f1,
                         'loss': epoch_loss/len(train_loader)})
        print(f"[{tag}] Epoch {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f} | F1: {f1:.4f}")

    elapsed = time.time() - start
    print(f"[{tag}] Total time: {elapsed:.0f}s")
    return history, elapsed

# Run full fine-tuning (transfer)
full_ft_history, full_ft_time = train_target(
    tgt_model, tgt_train_loader, tgt_val_loader, tag="Full-FT-Transfer"
)
# Run baseline (no transfer)
base_history, base_time = train_target(
    baseline_model, tgt_train_loader, tgt_val_loader, tag="No-Transfer-Baseline"
)

tgt_model.save_pretrained('/content/drive/MyDrive/NLP/Arabizi Tweets/models')

[Full-FT-Transfer] Epoch 1 | Loss: 0.9292 | F1: 0.5009
[Full-FT-Transfer] Epoch 2 | Loss: 0.7476 | F1: 0.6132
[Full-FT-Transfer] Epoch 3 | Loss: 0.6009 | F1: 0.6065
[Full-FT-Transfer] Total time: 138s
[No-Transfer-Baseline] Epoch 1 | Loss: 0.9703 | F1: 0.4149
[No-Transfer-Baseline] Epoch 2 | Loss: 0.7958 | F1: 0.6200
[No-Transfer-Baseline] Epoch 3 | Loss: 0.6513 | F1: 0.6091
[No-Transfer-Baseline] Total time: 137s


In [ ]:
!pip install tf-keras -q
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"  # tell transformers to use tf-keras not keras

import tensorflow as tf
from transformers import TFAutoModelForSequenceClassification
print("TF:", tf.__version__)

TF: 2.19.0


In [ ]:
import tensorflow as tf
from transformers import TFAutoModelForSequenceClassification
import numpy as np

# ── Build tf.data.Dataset from tokenized inputs ─────────────
def make_tf_dataset(texts, labels, tokenizer, batch=16, shuffle=True):
    enc = tokenizer(list(texts), truncation=True, padding=True,
                    max_length=128, return_tensors='np')
    ds = tf.data.Dataset.from_tensor_slices(
        ({'input_ids': enc['input_ids'],
          'attention_mask': enc['attention_mask']},
         np.array(list(labels)))
    )
    if shuffle:
        ds = ds.shuffle(1000)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

tf_train_ds = make_tf_dataset(tgt_train_df['clean_text'], tgt_train_df['label_id'], tokenizer)
tf_val_ds   = make_tf_dataset(tgt_val_df['clean_text'],   tgt_val_df['label_id'],   tokenizer, shuffle=False)

# ── Load TF version of source-trained checkpoint ────────────
# Note: HuggingFace saves in a format readable by both PT and TF
tf_model = TFAutoModelForSequenceClassification.from_pretrained(
    'bert-base-multilingual-cased',  # use base — TF loading from PT ckpt can be tricky
    num_labels=3,
    from_pt=True  # load PyTorch weights into TF model
)

# ── Keras compile: declarative vs PyTorch imperative ────────
tf_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# ── Train using Keras fit() — the key contrast with PyTorch ─
tf_start = time.time()
tf_history = tf_model.fit(
    tf_train_ds,
    validation_data=tf_val_ds,
    epochs=3,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)]
)
tf_time = time.time() - tf_start
print(f"TF training time: {tf_time:.0f}s")

tf_model.save_pretrained('/content/drive/MyDrive/NLP/Arabizi Tweets/models/tensorflow')

RuntimeError: Failed to import transformers.models.bert.modeling_tf_bert because of the following error (look up to see its traceback):
partially initialized module 'tf_keras.src' has no attribute 'utils' (most likely due to a circular import)

# ── LoRA (PEFT) Transfer Learning ──────────────────────────────\n# Instead of fine-tuning all 178M parameters, LoRA freezes the base model\n# and injects small trainable adapters (~0.33% of params) into attention layers.\n# Hypothesis: freezing base weights may preserve Arabic representations better\n# than full fine-tuning, leading to improved cross-script transfer.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSequenceClassification

# ── Load source checkpoint and swap to 3-class head ─────────
BASE = '/content/drive/MyDrive/NLP/Arabizi Tweets'

lora_base = AutoModelForSequenceClassification.from_pretrained(
    f'{BASE}/models',
    num_labels=3,
    ignore_mismatched_sizes=True
).to(device)

# ── Apply LoRA adapters to query and value projections ──────
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    bias="none"
)
lora_model = get_peft_model(lora_base, lora_config)
lora_model.print_trainable_parameters()
# Expected: ~0.33% of parameters trainable

In [ ]:
# ── Train LoRA model on target domain ────────────────────────
lora_history, lora_time = train_target(
    lora_model, tgt_train_loader, tgt_val_loader,
    epochs=3, lr=3e-4, tag="LoRA-Transfer"
)

# ── Save LoRA adapter weights ───────────────────────────────
lora_model.save_pretrained(f'{BASE}/models/lora')
print(f"✓ LoRA adapter saved to {BASE}/models/lora")

# ── Evaluation ─────────────────────────────────────────────────\n# Test set evaluation for all 3 PyTorch models:\n# 1. Full Fine-Tuning (transfer from MSA)\n# 2. No-Transfer Baseline (fresh mBERT)\n# 3. LoRA (transfer from MSA with frozen base + adapters)\n#\n# Metrics: classification report, confusion matrix, and\n# Dialectal Transfer Gap Score (DTGS) — measures F1 robustness\n# across varying levels of Arabizi intensity.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# ── Create test DataLoader ───────────────────────────────────
tgt_test_ds = ArabicTweetDataset(
    tgt_test_df['clean_text'], tgt_test_df['label_id'], tokenizer
)
tgt_test_loader = DataLoader(tgt_test_ds, batch_size=32, shuffle=False)

# ── Inference helper ─────────────────────────────────────────
def predict(model, loader):
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )
            all_preds.extend(out.logits.argmax(-1).cpu().tolist())
            all_true.extend(batch['labels'].tolist())
    return all_true, all_preds

# ── Run predictions for all 3 models ────────────────────────
LABEL_NAMES = ['Negative', 'Neutral', 'Positive']

models_dict = {
    'Full FT (Transfer)': tgt_model,
    'No-Transfer Baseline': baseline_model,
    'LoRA (Transfer)': lora_model,
}

results = {}
for name, model in models_dict.items():
    true, preds = predict(model, tgt_test_loader)
    results[name] = {'true': true, 'preds': preds}
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(classification_report(true, preds, target_names=LABEL_NAMES, digits=4))

In [ ]:
# ── Confusion Matrices ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(res['true'], res['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
    test_f1 = f1_score(res['true'], res['preds'], average='macro')
    ax.set_title(f'{name}\nTest F1: {test_f1:.4f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.savefig(f'{BASE}/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Confusion matrices saved")

In [ ]:
# ── Dialectal Transfer Gap Score (DTGS) ──────────────────────
# Measures how robust each model is across varying levels of
# Arabizi intensity (Latin character ratio in each tweet).
# Lower variance in F1 across quartiles = better cross-dialect transfer.

def latin_char_ratio(text):
    """Compute fraction of Latin characters in text as Arabizi intensity proxy."""
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    latin = sum(1 for c in text if c.isascii() and c.isalpha())
    total = sum(1 for c in text if c.isalpha())
    return latin / max(total, 1)

# Compute intensity for each test sample
test_texts = tgt_test_df['clean_text'].tolist()
test_labels = tgt_test_df['label_id'].tolist()
intensities = [latin_char_ratio(t) for t in test_texts]

# Split into 4 quartiles
quartile_labels = pd.qcut(intensities, q=4, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'])

print("Arabizi Intensity Distribution (Latin char ratio):")
print(f"  Q1 (Low):  {np.percentile(intensities, 25):.3f}")
print(f"  Q2 (Med):  {np.percentile(intensities, 50):.3f}")
print(f"  Q3:        {np.percentile(intensities, 75):.3f}")
print(f"  Q4 (High): {np.max(intensities):.3f}")
print()

# Compute F1 per quartile per model
dtgs_data = []
for name, res in results.items():
    quartile_f1s = []
    for q in ['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)']:
        mask = [ql == q for ql in quartile_labels]
        q_true = [t for t, m in zip(res['true'], mask) if m]
        q_pred = [p for p, m in zip(res['preds'], mask) if m]
        if len(set(q_true)) > 1:
            qf1 = f1_score(q_true, q_pred, average='macro')
        else:
            qf1 = float('nan')
        quartile_f1s.append(qf1)
        dtgs_data.append({'Model': name, 'Quartile': q, 'F1': qf1, 'N': sum(mask)})

    variance = np.nanvar(quartile_f1s)
    print(f"{name}: DTGS variance = {variance:.6f} (lower = more robust)")

dtgs_df = pd.DataFrame(dtgs_data)
print("\n── DTGS Quartile Breakdown ──")
print(dtgs_df.pivot(index='Quartile', columns='Model', values='F1').round(4).to_string())

In [ ]:
# ── DTGS Visualization ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
pivot = dtgs_df.pivot(index='Quartile', columns='Model', values='F1')
pivot.plot(kind='bar', ax=ax, rot=0)
ax.set_ylabel('Macro F1')
ax.set_title('Dialectal Transfer Gap Score (DTGS)\nF1 by Arabizi Intensity Quartile')
ax.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(f'{BASE}/dtgs_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary Comparison Table ─────────────────────────────────
print("\n── Final Model Comparison ──")
summary_rows = []
for name, res in results.items():
    test_f1 = f1_score(res['true'], res['preds'], average='macro')
    quartile_f1s = []
    for q in ['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)']:
        mask = [ql == q for ql in quartile_labels]
        q_true = [t for t, m in zip(res['true'], mask) if m]
        q_pred = [p for p, m in zip(res['preds'], mask) if m]
        if len(set(q_true)) > 1:
            quartile_f1s.append(f1_score(q_true, q_pred, average='macro'))
    dtgs_var = np.nanvar(quartile_f1s)
    summary_rows.append({
        'Model': name,
        'Test F1': round(test_f1, 4),
        'DTGS Variance': round(dtgs_var, 6),
    })

# Add training time and param info
time_map = {'Full FT (Transfer)': full_ft_time, 'No-Transfer Baseline': base_time, 'LoRA (Transfer)': lora_time}
param_map = {'Full FT (Transfer)': '~178M (100%)', 'No-Transfer Baseline': '~178M (100%)', 'LoRA (Transfer)': '~590K (~0.33%)'}
for row in summary_rows:
    row['Train Time (s)'] = round(time_map[row['Model']], 0)
    row['Trainable Params'] = param_map[row['Model']]

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# ── Gradio Demo ────────────────────────────────────────────────\n# Interactive interface for testing all 3 models on custom Arabizi text.\n# Outputs: predicted sentiment, confidence scores, Arabizi intensity.

In [ ]:
import gradio as gr
from peft import PeftModel

BASE = '/content/drive/MyDrive/NLP/Arabizi Tweets'

# ── Load all 3 saved models ─────────────────────────────────
# Full FT (transfer) — already loaded as tgt_model, but reload from disk for safety
demo_full_ft = AutoModelForSequenceClassification.from_pretrained(
    f'{BASE}/models', num_labels=3, ignore_mismatched_sizes=True
).to(device).eval()

# Baseline — fresh mBERT (no source training)
# Note: baseline wasn't saved separately, so we use the in-memory model
demo_baseline = baseline_model.eval()

# LoRA — load base + adapter
lora_base_for_demo = AutoModelForSequenceClassification.from_pretrained(
    f'{BASE}/models', num_labels=3, ignore_mismatched_sizes=True
).to(device)
demo_lora = PeftModel.from_pretrained(lora_base_for_demo, f'{BASE}/models/lora').to(device).eval()

demo_models = {
    'Full FT (Transfer)': demo_full_ft,
    'No-Transfer Baseline': demo_baseline,
    'LoRA (Transfer)': demo_lora,
}

LABEL_NAMES = ['Negative', 'Neutral', 'Positive']

def predict_sentiment(text, model_choice):
    cleaned = clean_arabizi(text)
    intensity = latin_char_ratio(cleaned)

    enc = tokenizer(cleaned, return_tensors='pt', truncation=True,
                    padding=True, max_length=128)
    enc = {k: v.to(device) for k, v in enc.items()}

    model = demo_models[model_choice]
    with torch.no_grad():
        logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    sentiment = LABEL_NAMES[probs.argmax()]
    confidence = {LABEL_NAMES[i]: float(probs[i]) for i in range(3)}

    return (
        f"**{sentiment}** (confidence: {probs.max():.2%})",
        confidence,
        f"{intensity:.2%}"
    )

demo = gr.Interface(
    fn=predict_sentiment,
    inputs=[
        gr.Textbox(label="Enter Arabizi text", placeholder="e.g. hayda ktir mni7!"),
        gr.Dropdown(choices=list(demo_models.keys()), value='Full FT (Transfer)', label="Model")
    ],
    outputs=[
        gr.Textbox(label="Predicted Sentiment"),
        gr.Label(label="Confidence Scores", num_top_classes=3),
        gr.Textbox(label="Arabizi Intensity (Latin char ratio)")
    ],
    title="Arabizi Sentiment Analyzer",
    description="Classify Lebanese Arabizi tweets as Positive, Neutral, or Negative using transfer learning from MSA Arabic.",
)

demo.launch(share=True)